# DQN only Evaluation

In [ ]:
import sys
import os
import torch
import pandas as pd

if 'google.colab' in sys.modules:
  from google.colab import drive
  drive.mount( "/content/drive")
  if os.path.isdir('drive/MyDrive/Projects/Offline_RL_BSc_Thesis/notebooks/DQN/DQN_only'):
    os.chdir('drive/MyDrive/Projects/Offline_RL_BSc_Thesis/notebooks/DQN/DQN_only')


project_root = os.path.abspath(os.path.join(os.path.dirname("__file__"), "../../../../"))
if project_root not in sys.path:
    sys.path.append(project_root)

torch_device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

selected_features = ['X', 'Y', 'lv_X', 'lv_Y', 'reward', 'angle', 'angular_velocity', 'leg_1', 'leg_2']

## Test Evaluation

In [ ]:
from src.utils.evaluation import print_model_eval_results, evaluate_model_test_dataset
from src.utils.plotting import plot_confusion_matrix_heatmap

### Replay Buffer

In [ ]:
rb_norm_name = 'standard'

rb_test_df = pd.read_parquet('../../../data/replay_buffer_episodes/rb_test.parquet').drop(columns=['done', 'episode', 'reward'])
rb_normalization_technique = torch.jit.load(f'../../../models/BC/replay_buffer/normalization/{rb_norm_name}_normalization.pt')
#rb_normalization_technique = None
rb_model = torch.jit.load(f'../../../models/DQN/replay_buffer/DQN_only_{rb_norm_name}.pt')

rb_classificiation_repport_dict, rb_confusion_matrix = evaluate_model_test_dataset(model=rb_model,
                            device=torch_device,
                            test_df=rb_test_df,
                            norm_technique_script=rb_normalization_technique,
                            selected_features=selected_features,
                            apply_softmax=False)

rb_confusion_matrix_fig = plot_confusion_matrix_heatmap(confusion_matrix=rb_confusion_matrix,
                                                        model_name='BC Replay Buffer',
                                                        f_size=(5,5))

print_model_eval_results(rb_classificiation_repport_dict,
                         model_name='BC Replay Buffer')

### Final Policy

In [ ]:
fp_norm_name = 'max_abs'

fp_test_df = pd.read_parquet('../../../data/final_policy_episodes/fp_test.parquet').drop(columns=['done', 'episode', 'reward'])
fp_normalization_technique = torch.jit.load(f'../../../models/BC/final_policy/normalization/{fp_norm_name}_normalization.pt')
#fp_normalization_technique = None
fp_model = torch.jit.load(f'../../../models/DQN/final_policy/DQN_only_{fp_norm_name}.pt')


fp_classificiation_repport_dict, fp_confusion_matrix = evaluate_model_test_dataset(model=fp_model,
                            device=torch_device,
                            test_df=fp_test_df,
                            norm_technique_script=fp_normalization_technique,
                            selected_features=selected_features,
                            apply_softmax=False)

fp_confusion_matrix_fig = plot_confusion_matrix_heatmap(confusion_matrix=fp_confusion_matrix,
                                                        model_name='BC Final Policy',
                                                        f_size=(5,5))

print_model_eval_results(fp_classificiation_repport_dict,
                         model_name='BC Final Policy')

## Evaluation in the Live Environment for 1000 episodes

In [ ]:
import yaml
import warnings

from src.utils.live_env_testing import DQN_BC_evaluate_model_in_live_env

warnings.filterwarnings("ignore")
with open('../../../config/DQN/DQN_only/dqn_only_live_env_elavuation_config.yaml', 'r') as f:
    dqn_only_live_env_config_config = yaml.safe_load(f)

### Replay Buffer

In [ ]:
rb_rewards = DQN_BC_evaluate_model_in_live_env(
    env_test_params=dqn_only_live_env_config_config,
    model_name='replay_buffer',
    norm_technique=rb_normalization_technique,
    model = rb_model
)

### Final Policy

In [ ]:
fp_rewards = DQN_BC_evaluate_model_in_live_env(
    env_test_params=dqn_only_live_env_config_config,
    norm_technique=fp_normalization_technique,
    model_name='final_policy',
    model = fp_model
)

### Accumulated rewards distribution comparison

In [ ]:
from src.utils.plotting import plot_univariate_analysis


rb_rewards_df = pd.DataFrame(rb_rewards, columns=['reward'])
fp_rewards_df = pd.DataFrame(fp_rewards, columns=['reward'])

rb_fp_combined_fig = plot_univariate_analysis(
        df1_name = 'rb_agent',
        df2_name = 'fp_agent',
        df1=rb_rewards_df,
        df2=fp_rewards_df,
        num_columns=3,
        custom_title='DQN-only Accumulated Reward Distribution After 1000 Episodes (Live)',
        kde_kwargs={'linewidth': 2, 'alpha': 0.5},
        f_size=(16,4))
rb_fp_combined_fig.savefig('../../../plots/DQN_only_liven_env_1000_ep_eval_rb_fp.jpg')